In [16]:
# Import live code changes in
%load_ext autoreload
%autoreload 

import os

from pathlib import Path
import xarray as xr
import scipy.stats as stats
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings 
warnings.filterwarnings("ignore", category=RuntimeWarning)

from sovereign.utils import pot_with_optimal_threshold

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### Set filepaths and data info

In [2]:
root = Path.cwd().parent # find project root
isimip_path = Path(os.path.join(root, 'inputs', 'flood', 'future', 'isimip', 'UGA'))
prelim_output_path = Path(os.path.join(root, 'outputs', 'flood', 'future', 'prelim')) 
final_output_path = Path(os.path.join(root, 'outputs', 'flood', 'future'))
return_periods = [5, 10, 20, 25, 50, 75, 100, 200, 250, 500, 1000] 
sample_periods = {
    "historical": ("1965-01-01", "2014-12-31"),
    "2030": ("2005-01-01", "2054-12-31"),
    "2040": ("2015-01-01", "2064-12-31"),
    "2050": ("2025-01-01", "2074-12-31"),
    "2060": ("2035-01-01", "2084-12-31"),
    "2070": ("2045-01-01", "2094-12-31")
}

#### Extract point-model results

In [3]:
output = pd.DataFrame()
models = [f for f in os.listdir(isimip_path) if f.endswith('.nc')]

for model in models:
    if os.path.exists(os.path.join(prelim_output_path, f'{model}.parquet')):
        print(f"Skipping model {model}, results already exist.")
        output_existing = pd.read_parquet(os.path.join(prelim_output_path, f'{model}.parquet'))
        output = pd.concat([output, output_existing], ignore_index=True)
        continue
    model_output = pd.DataFrame()
    print(f"Processing model: {model}")
    dataset = xr.open_dataset(os.path.join(isimip_path, f'{model}'))
    latitudes = dataset['lat'].values
    longitudes = dataset['lon'].values
    grid_points = [(lat, lon) for lat in latitudes for lon in longitudes]
    climate_scenario = model.split('_')[-1].replace('.nc', '')

    for cell in tqdm(grid_points, total=len(grid_points)):
        lat = cell[0]
        lon = cell[1]
        dataset_sample = dataset.sel(lat=lat, lon=lon, method="nearest")
        for period_name, (start_date, end_date) in sample_periods.items():
            period_data =  dataset_sample.sel(time=slice(start_date, end_date))['dis']
            for return_period in return_periods:
                if np.std(period_data.values) == 0.0:
                    continue  # skip locations with no variability
                try:
                    best, diagnostics = pot_with_optimal_threshold(
                        x=period_data.values,
                        candidate_ps=[0.90, 0.92, 0.94, 0.96, 0.97, 0.98, 0.99],
                        return_period=return_period
                    )
                    if best is None:
                        continue  # no suitable threshold found
                    new_output = pd.DataFrame({
                        "latitude": lat,
                        "longitude": lon,
                        "return_level": best['qT'],
                        "log_return_level": best['log_qT'],
                        "std_log_return_level": best['var_log_qT'] ** 0.5,
                        "q5": np.exp(best['log_qT'] - 1.96 * best['var_log_qT'] ** 0.5),
                        "q95": np.exp(best['log_qT'] + 1.96 * best['var_log_qT'] ** 0.5),
                        "xi_hat": best['xi'],
                        "sigma_hat": best['sigma'],
                        "selected_threshold": best['p'],
                        "threshold_discharge": best['u'],
                        "num_exceedances": best['n_exc'],
                        "ks_pvalue": best['ks_pvalue'],
                        "return_period": return_period,
                        "model": model,
                        "period": period_name,
                        'period_start_date': start_date,
                        'period_end_date': end_date,
                        "climate_scenario": climate_scenario,
                    }, index=[0])
                    model_output = pd.concat([model_output, new_output], ignore_index=True)
                except Exception as e:
                    print(f"Error at lat {lat}, lon {lon}: {e}")
    model_output.to_parquet(os.path.join(prelim_output_path, f'{model}.parquet'))
    output = pd.concat([output, model_output], ignore_index=True)
output = output.dropna()
output.to_parquet(os.path.join(prelim_output_path, 'extracted_point_model_results_2.parquet'))

Skipping model UGA_dis_jules-w2_ukesm1-0-ll_ssp585.nc, results already exist.
Skipping model UGA_dis_jules-w2_mri-esm2-0_ssp585.nc, results already exist.
Skipping model UGA_dis_jules-w2_mpi-esm1-2-hr_ssp585.nc, results already exist.
Skipping model UGA_dis_jules-w2_ipsl-cm6a-lr_ssp585.nc, results already exist.
Skipping model UGA_dis_jules-w2_gfdl-esm4_ssp585.nc, results already exist.


In [6]:
baseline_levels = output[output['period'] == 'historical'][[
    'latitude', 'longitude', 'climate_scenario', 'model', 'return_period', 'return_level', 'q5', 'q95', 'xi_hat', 'sigma_hat', 'selected_threshold']
    ]

In [10]:
baseline_levels

,latitude,longitude,climate_scenario,model,return_period,return_level,q5,q95,xi_hat,sigma_hat,selected_threshold
0,5.75,28.25,ssp585,UGA_dis_jules-w2_ukesm1-0-ll_ssp585.nc,5,17.043119,16.502991,17.600924,0.219019,10.455960,0.90
1,5.75,28.25,ssp585,UGA_dis_jules-w2_ukesm1-0-ll_ssp585.nc,10,23.767964,23.767646,23.768282,0.219019,10.455960,0.90
2,5.75,28.25,ssp585,UGA_dis_jules-w2_ukesm1-0-ll_ssp585.nc,20,31.595295,31.130614,32.066911,0.219019,10.455960,0.90
3,5.75,28.25,ssp585,UGA_dis_jules-w2_ukesm1-0-ll_ssp585.nc,25,34.378704,33.772900,34.995375,0.219019,10.455960,0.90
4,5.75,28.25,ssp585,UGA_dis_jules-w2_ukesm1-0-ll_ssp585.nc,50,43.945583,42.850308,45.068853,0.219019,10.455960,0.90
...,...,...,...,...,...,...,...,...,...,...,...
101492,-2.75,36.75,ssp585,UGA_dis_jules-w2_gfdl-esm4_ssp585.nc,100,2452.706080,2373.421914,2534.638734,0.032425,712.094941,0.94
101493,-2.75,36.75,ssp585,UGA_dis_jules-w2_gfdl-esm4_ssp585.nc,200,2981.746131,2865.237532,3102.992298,0.032425,712.094941,0.94
101494,-2.75,36.75,ssp585,UGA_dis_jules-w2_gfdl-esm4_ssp585.nc,250,3154.604709,3021.143865,3293.961267,0.032425,712.094941,0.94
101495,-2.75,36.75,ssp585,UGA_dis_jules-w2_gfdl-esm4_ssp585.nc,500,3699.598995,3494.477148,3916.761262,0.032425,712.094941,0.94


In [8]:
output_with_baseline =pd.merge(
    output,
    baseline_levels,
    on=['latitude', 'longitude', 'climate_scenario', 'model', 'return_period'],
    suffixes=('', '_baseline')
)

output_with_baseline['multiplier'] = output_with_baseline['return_level'] / output_with_baseline['return_level_baseline']
output_with_baseline['multiplier_q5'] = output_with_baseline['q5'] / output_with_baseline['return_level_baseline']
output_with_baseline['multiplier_q95'] = output_with_baseline['q95'] / output_with_baseline['return_level_baseline']

In [11]:
output_with_baseline

,latitude,longitude,return_level,log_return_level,std_log_return_level,q5,q95,xi_hat,sigma_hat,selected_threshold,...,climate_scenario,return_level_baseline,q5_baseline,q95_baseline,xi_hat_baseline,sigma_hat_baseline,selected_threshold_baseline,multiplier,multiplier_q5,multiplier_q95
0,5.75,28.25,17.043119,2.835747,0.016431,16.502991,17.600924,0.219019,10.45596,0.90,...,ssp585,17.043119,16.502991,17.600924,0.219019,10.455960,0.90,1.000000,0.968308,1.032729
1,5.75,28.25,23.767964,3.168339,0.000007,23.767646,23.768282,0.219019,10.45596,0.90,...,ssp585,23.767964,23.767646,23.768282,0.219019,10.455960,0.90,1.000000,0.999987,1.000013
2,5.75,28.25,31.595295,3.453008,0.007559,31.130614,32.066911,0.219019,10.45596,0.90,...,ssp585,31.595295,31.130614,32.066911,0.219019,10.455960,0.90,1.000000,0.985293,1.014927
3,5.75,28.25,34.378704,3.537437,0.009071,33.772900,34.995375,0.219019,10.45596,0.90,...,ssp585,34.378704,33.772900,34.995375,0.219019,10.455960,0.90,1.000000,0.982379,1.017938
4,5.75,28.25,43.945583,3.782952,0.012877,42.850308,45.068853,0.219019,10.45596,0.90,...,ssp585,43.945583,42.850308,45.068853,0.219019,10.455960,0.90,1.000000,0.975077,1.025560
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99951,-2.75,36.75,3340.007850,8.113728,0.020233,3210.143842,3475.125412,0.150807,903.12841,0.92,...,ssp585,2452.706080,2373.421914,2534.638734,0.032425,712.094941,0.94,1.361764,1.308817,1.416854
99952,-2.75,36.75,4242.961096,8.353017,0.026207,4030.519113,4466.600543,0.150807,903.12841,0.92,...,ssp585,2981.746131,2865.237532,3102.992298,0.032425,712.094941,0.94,1.422979,1.351731,1.497982
99953,-2.75,36.75,4554.314025,8.423830,0.028737,4304.883188,4818.197226,0.150807,903.12841,0.92,...,ssp585,3154.604709,3021.143865,3293.961267,0.032425,712.094941,0.94,1.443704,1.364635,1.527354
99954,-2.75,36.75,5591.071851,8.628926,0.038136,5188.399255,6024.995940,0.150807,903.12841,0.92,...,ssp585,3699.598995,3494.477148,3916.761262,0.032425,712.094941,0.94,1.511264,1.402422,1.628554


In [12]:
def get_adjusted_return_period(
    xi_hat,
    sigma_hat,
    threshold_discharge,  
    return_level_scenario,
    return_level_baseline,
    baseline_return_period
):
    # Basic checks
    vals = [xi_hat, sigma_hat, threshold_discharge, return_level_scenario, return_level_baseline, baseline_return_period]
    if any(v is None or np.isnan(v) for v in vals):
        return np.nan
    if sigma_hat <= 0 or baseline_return_period <= 0:
        return np.nan

    if abs(xi_hat) < 1e-6:
        return float(baseline_return_period * np.exp((return_level_baseline - return_level_scenario) / sigma_hat))

    # ξ ≠ 0: general GP case
    num = 1 + xi_hat * (return_level_baseline - threshold_discharge) / sigma_hat
    den = 1 + xi_hat * (return_level_scenario - threshold_discharge) / sigma_hat

    # Domain check for the GP (must be > 0)
    if num <= 0 or den <= 0:
        return np.nan

    ratio = (num / den) ** (1.0 / xi_hat)
    return float(baseline_return_period * ratio)

In [13]:
output_with_baseline['adjusted_return_period'] = output_with_baseline.apply(
    lambda row: get_adjusted_return_period(
        xi_hat=row['xi_hat'],
        sigma_hat=row['sigma_hat'],
        threshold_discharge=row['selected_threshold'],
        return_level_scenario=row['return_level'],
        return_level_baseline=row['return_level_baseline'],
        baseline_return_period=row['return_period']
    ),
    axis=1
)

output_with_baseline['adjusted_return_period_q5'] = output_with_baseline.apply(
    lambda row: get_adjusted_return_period(
        xi_hat=row['xi_hat'],
        sigma_hat=row['sigma_hat'],
        threshold_discharge=row['selected_threshold'],
        return_level_scenario=row['q5'],
        return_level_baseline=row['return_level_baseline'],
        baseline_return_period=row['return_period']
    ),
    axis=1
)

output_with_baseline['adjusted_return_period_q95'] = output_with_baseline.apply(
    lambda row: get_adjusted_return_period(
        xi_hat=row['xi_hat'],
        sigma_hat=row['sigma_hat'],
        threshold_discharge=row['selected_threshold'],
        return_level_scenario=row['q95'],
        return_level_baseline=row['return_level_baseline'],
        baseline_return_period=row['return_period']
    ),
    axis=1
)

In [15]:
output_with_baseline.to_parquet(os.path.join(final_output_path, 'results.parquet.gzip'), index=False, compression='gzip')